# 00 · Data Ingestion (2015–2025 Raw Datasets)

**Project:** AgriShield — Early Warning System for Crop Yield & Livestock Forage Risk in Kenya  
**Notebook Goal:** Load and inspect our **10 raw datasets (2015–2025)** from `data_pipeline/data/` before cleaning them in the next notebook.

### Data Sources (Proposal Table 3.1)
1. **Geospatial & County Codes:** `kenya_counties.geojson` and `kenya_county_crosswalk.csv` (all 47 Kenyan counties).
2. **Forage & Vegetation (NASA MODIS):** `ken_ndvi_subnational_2015_2025_raw.csv` (pasture greenness / NDVI).
3. **Weather & Soil Moisture (CHIRPS & ERA5-Land):** Rainfall, temperature, and root-zone soil moisture (`2015–2025`).
4. **Crop Production & Demographics (KNBS & FAOSTAT):** Historical crop harvests (`2015–2025`) and county farming households (`2019 Census`).

---
### Step 1: Import Libraries & Check Our 10 Raw Files
In the cell below, we import `pandas`, `numpy`, `json`, and `Path`, point to our `../data` folder, and check that all 10 raw files are present.

In [1]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Point to the folder where our 10 raw datasets are stored
DATA_DIR = Path("../data")

# List all files in the data folder and check their sizes
files = sorted(DATA_DIR.glob("*"))

file_summary = pd.DataFrame([
    {
        "File Name": f.name,
        "Size (KB)": round(f.stat().st_size / 1024, 1)
    }
    for f in files if not f.name.startswith(".")
])

print(f"Total raw files found: {len(file_summary)}")
display(file_summary)

Total raw files found: 4


,File Name,Size (KB)
0,external,0.0
1,kenya_county_crosswalk.csv,1.4
2,processed,0.0
3,raw,0.4


### Step 2: Load Geospatial Boundaries & the 47-County Crosswalk

Why do we need these two files?
1. **`kenya_counties.geojson`**: Contains the map boundaries (polygons) for all **47 counties in Kenya**.
2. **`kenya_county_crosswalk.csv`**: Links different county IDs (`knbs_code`, `pcode` like `KE001`–`KE047`, and uppercase names) to the standard names of all **47 Kenyan counties** so we can merge all datasets accurately.

In [2]:
# 1. Load the 47-county crosswalk CSV

crosswalk_df = pd.read_csv(DATA_DIR / "kenya_county_crosswalk.csv")
print(f"Crosswalk shape: {crosswalk_df.shape} (Total counties: {crosswalk_df['county'].nunique()})")
display(crosswalk_df.head(10))

# 2. Load the Kenya Counties GeoJSON and verify all 47 counties are present

with open(DATA_DIR / "kenya_counties.geojson", "r") as f:
    kenya_geojson = json.load(f)

geojson_counties = sorted([feat["properties"]["county"] for feat in kenya_geojson["features"]])
print(f"\nTotal county polygons in GeoJSON: {len(geojson_counties)}")
print("First 10 counties in GeoJSON:", geojson_counties[:10])

Crosswalk shape: (47, 5) (Total counties: 47)


,county,knbs_code,pcode,desinventar_code,desinventar_name
0,Mombasa,1,KE001,28,MOMBASA
1,Kwale,2,KE002,19,KWALE
2,Kilifi,3,KE003,14,KILIFI
3,Tana River,4,KE004,40,TANA-RIVER
4,Lamu,5,KE005,21,LAMU
5,Taita-Taveta,6,KE006,39,TAITA-TAVETA
6,Garissa,7,KE007,6,GARISSA
7,Wajir,8,KE008,46,WAJIR
8,Mandera,9,KE009,24,MANDERA
9,Marsabit,10,KE010,25,MARSABIT


FileNotFoundError: [Errno 2] No such file or directory: '../data/kenya_counties.geojson'

### Step 3: Load Raw NASA MODIS NDVI & CHIRPS Rainfall Data (2015–2025)

Next, we load our two main satellite time-series datasets from `2015` to `2025`:
1. **`ken_ndvi_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **Normalized Difference Vegetation Index (NDVI)** from NASA MODIS (`vim` = vegetation greenness, `viq` = vegetation anomaly %). This measures pasture and crop health across all 47 counties.
2. **`ken_chirps_rainfall_subnational_2015_2025_raw.csv`**: Raw 10-day (dekadal) **CHIRPS v2.0 Rainfall** (`rfh` = 10-day rainfall in mm, `r1h` = 1-month rolling rainfall, `r3h` = 3-month rolling rainfall, `rfq` = rainfall anomaly %).

Notice that both files are raw and uncleaned: they contain a metadata tag row at the top, use `PCODE`s instead of county names, and include both county (`adm_level = 1`) and sub-county (`adm_level = 2`) records. We inspect these raw issues here so we know what to clean in `01_data_cleaning.ipynb`.

In [ ]:
# 1. Load raw NASA MODIS NDVI (2015-2025

ndvi_raw = pd.read_csv(DATA_DIR / "ken_ndvi_subnational_2015_2025_raw.csv", low_memory=False)
print("====== Raw NASA MODIS NDVI (2015-2025) ======")
print(f"Shape: {ndvi_raw.shape}")
print(f"Columns: ", list(ndvi_raw.columns))
display(ndvi_raw.head(10))

# 2. Load raw CHIRPS Rainfall (2015-2025)

chirps_raw = pd.read_csv(DATA_DIR / "ken_chirps_rainfall_subnational_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw CHIRPS Subnational Rainfall (2015-2025) ======")
print(f"Shape: {chirps_raw.shape}")
print(f"Columns: ", list(chirps_raw.columns))
display(chirps_raw.head(10))
d


### Step 4: Load Raw ERA5-Land Soil Moisture, Precipitation & MODIS Temperature Data (2015–2025)

In this step, we load the three meteorological and land-surface datasets that provide **temperature** and **root-zone soil moisture** across all **47 counties** from `2015` to `2025`:
1. **`era5land_soil_moisture_runoff_2015_2025_raw.csv`**: Raw Google Earth Engine (`ECMWF/ERA5-Land`) monthly county data containing `volumetric_soil_water_layer_1` (`0–7 cm` top-soil moisture) and `volumetric_soil_water_layer_2` (`7–28 cm` root-zone moisture).
2. **`era5land_precipitation_2015_2025_raw.csv`**: Raw `ERA5-Land` monthly precipitation (`mean`) across all 47 counties.
3. **`modis_era5_climate_vegetation_monthly_2015_2025_raw.csv`**: Raw monthly grid observations containing `max_temp_c`, `min_temp_c`, `mean_temp_c`, `humidity_rh_pct`, `soil_moisture_m3m3`, `rain_mm`, and `ndvi` (`2015–2025`).

**Raw Data Issues to Note for Cleaning Later:**
- The Earth Engine files store county names in uppercase under the column `FIRST_COUN` and include unnecessary GEE system columns (`system:index`, `COUNT`, `.geo`).
- The monthly grid file contains multiple `(lat, lon)` points per county and some border points labeled `"Unknown"`.

In [ ]:
# 1. Load raw ERA5-Land Soil Moisture & Runoff (2015-2025)

era5_soil_raw = pd.read_csv(DATA_DIR / "era5land_soil_moisture_runoff_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Soil Moisture & Runoff (2015-2025) ======")
print(f"Shape: {era5_soil_raw.shape} | Unique Counties: {era5_soil_raw['FIRST_COUN'].nunique()}")
display(era5_soil_raw.head(10))

# 2. Load raw ERA5-Land Monthly Precipitation (2015-2025)

era5_precip_raw = pd.read_csv(DATA_DIR / "era5land_precipitation_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw ERA5-Lnd Monthly Precipitation (2015-2025) ======")
print(f"Shape: {era5_precip_raw.shape} | Unique Counties: {era5_precip_raw['FIRST_COUN'].nunique()}")
display(era5_precip_raw.head(10))

# 3. Load raw ERA5-Land Monthly Temperature (2015-2025)

climate_grid_raw = pd.read_csv(DATA_DIR / "modis_era5_climate_vegetation_monthly_2015_2025_raw.csv", low_memory=False)
print("\n====== Raw MODIS-ERA5 Climate & Vegetation (2015-2025) ======")
print(f"Shape: {climte_grid_raw.shape} | Data Range: {climate_grid_raw['month'].min()} to {climate_grid_raw['month'].max()}")
print("Columns: ", list(climate_grid_raw.columns))
display(climate_grid_raw.head(10))

### Step 5: Load Raw Crop Production (FAOSTAT & KNBS) & County Demographic Data (2015–2025)

Next, we load the three socioeconomic and agricultural production datasets from **Table 3.1** of our Project Proposal:
1. **`faostat_kenya_crop_production_2015_2025_raw.csv`**: Annual crop yield (`kg/ha`), production (`tonnes`), and area harvested (`ha`) from **FAOSTAT (`2015–2025`)**.
2. **`knbs_agricultural_production_2015_2025_raw.csv`**: Marketed and physical agricultural & livestock output from the **KNBS Statistical Abstract & Economic Survey Chapter 7 (`2015–2025*`)**.
3. **`knbs_2019_census_agriculture_households_raw.csv`**: County-level distribution of households practicing farming, crop production, livestock rearing, aquaculture, fishing, and irrigation from the **KNBS 2019 Census (Volume IV, Table 2.19)**.

**Raw Data Issues to Note for Cleaning Later:**
- The FAOSTAT file includes neighboring East African countries (`Uganda`, `Ethiopia`, `Tanzania`) alongside `Kenya`, plus flag columns (`A`, `E`).
- The KNBS production and census tables store numbers as comma-formatted text strings (e.g., `"8,506.0"`, `"12,043,016"`), have provisional column headers (`2024*`, `2025*`), and include a national `"KENYA"` total row (`County_Code = 000`) and slash/hyphen county names (`TAITA/TAVETA`, `ELGEYO/MARAKWET`).

In [ ]:
# 1. Load raw FAOSTAT Crop Production, Yield & Area Harvested (2015-2025)

faostat_raw = pd.read_csv(DATA_DIR / "faostat_kenya_crop_production_2015_2025_raw.csv")
print("=== 1. Raw FAOSTAT Crop Production & Yield (2015–2025) ===")
print(f"Shape: {faostat_raw.shape} | Countries in file: {faostat_raw['Country'].unique().tolist()}")
display(faostat_raw.head(5))

# 2. Load raw KNBS Chapter 7 Agricultural & Livestock Production (2015-2025*)

knbs_prod_raw = pd.read_csv(DATA_DIR / "knbs_agricultural_production_2015_2025_raw.csv")
print("\n=== 2. Raw KNBS Agricultural & Livestock Production (2015–2025*) ===")
print(f"Shape: {knbs_prod_raw.shape}")
display(knbs_prod_raw.head(6))

# 3. Load raw KNBS 2019 Census Agricultural & Livestock Households by County

census_demo_raw = pd.read_csv(DATA_DIR / "knbs_2019_census_agriculture_households_raw.csv")
print("\n=== 3. Raw KNBS 2019 Census Agricultural Households ===")
print(f"Shape: {census_demo_raw.shape} (Includes 1 national 'KENYA' total row + 47 counties)")
display(census_demo_raw.head(6))

### Step 6: Raw Data Quality & Audit Summary (What We Need to Clean Next)

Before moving to **`01_data_cleaning.ipynb`**, we generate a single **Raw Data Audit Summary Table** checking the rows, columns, missing values, and specific cleaning tasks required for each dataset.

In [ ]:
# Build a simple, clear audit summary of all tabular raw datasets
raw_datasets = {
    "County Crosswalk": (crosswalk_df, "Reference lookup for 47 counties (PCODE & KNBS codes)"),
    "MODIS NDVI (2015–2025)": (ndvi_raw, "Filter adm_level==1, map PCODE to county, aggregate dekads to monthly mean"),
    "CHIRPS Rainfall (2015–2025)": (chirps_raw, "Filter adm_level==1, map PCODE to county, sum dekadal rainfall per month"),
    "ERA5 Soil Moisture (2015–2025)": (era5_soil_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "ERA5 Precipitation (2015–2025)": (era5_precip_raw, "Map uppercase FIRST_COUN to standard county, drop GEE system/.geo columns"),
    "MODIS/ERA5 Climate Grid (2015–2025)": (climate_grid_raw, "Drop 'Unknown' border points, standardize county names, average per county-month"),
    "FAOSTAT Crops (2015–2025)": (faostat_raw, "Filter Country=='Kenya', keep clean numeric Yield & Production columns"),
    "KNBS Production (2015–2025*)": (knbs_prod_raw, "Strip commas from numbers, rename '2024*'/'2025*' to '2024'/'2025', reshape wide/long"),
    "KNBS 2019 Census Households": (census_demo_raw, "Drop 'KENYA' total row (000), strip commas, standardize 47 county names"),
}

audit_rows = []
for name, (df, cleaning_task) in raw_datasets.items():
    audit_rows.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Action Required in 01_data_cleaning.ipynb": cleaning_task,
    })

audit_df = pd.DataFrame(audit_rows)
display(audit_df)
print("\nNotebook 00 (Data Ingestion) Complete! Ready for 01_data_cleaning.ipynb.")